In [1]:
# Run everytime a new function is added
import numpy as np
from scripts.utilities import *
from scripts.features import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn import metrics

ModuleNotFoundError: No module named 'xgboost'

In [2]:
consumer_df, account_df, transaction_df = get_data()
transaction_df.amount = transaction_df.amount.apply(abs)

Data successfully loaded and processed.


In [3]:
# c_df = consumer_df.dropna(subset='DQ_TARGET')
# a_df = account_df[account_df.prism_consumer_id.isin(c_df.prism_consumer_id)]
# t_df = transaction_df[transaction_df.prism_consumer_id.isin(c_df.prism_consumer_id)]

In [ ]:
c_df = consumer_df
a_df = account_df
t_df = transaction_df

## Account balance overtime:

- Balance recorded at the time account_df was made:

In [4]:
balance = a_df.groupby(['prism_consumer_id']).agg({'balance_date':'max', 'balance':'sum'})
# display(balance)
acct_balance = balance['balance']
# acct_balance

- Current balance:

In [5]:
t = t_df.copy()

In [6]:
t['balance_date'] = balance['balance_date']
t['amount'] = np.where(t['credit_or_debit'] == 'DEBIT', -t['amount'], t['amount'])
t['is_before_balance_date'] = np.where(t['posted_date'] < t['balance_date'], True, False)
update_balance = t[t['is_before_balance_date'] == False]
update_balance = update_balance.groupby(['prism_consumer_id'])['amount'].sum()

In [7]:
current_balance = c_df[['prism_consumer_id']]

current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)
current_balance['current_balance'] = current_balance['balance'] + current_balance['update']
current_balance = current_balance.current_balance
# current_balance

C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\921156377.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\921156377.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)


## Spending overtime:

In [8]:
inflows = t_df[t_df.credit_or_debit == 'CREDIT']
outflows = t_df[t_df.credit_or_debit == 'DEBIT']

# display(inflows, outflows)
inflows.shape, outflows.shape

((878829, 6), (4258005, 6))

In [9]:
avg_spending = outflows.groupby('prism_consumer_id')['amount'].mean()
# avg_spending

In [10]:
outflows['year'] = outflows['posted_date'].dt.year
outflows['month'] = outflows['posted_date'].dt.month
outflows['week'] = outflows['posted_date'].dt.isocalendar().week
monthly_totals = outflows.groupby(['prism_consumer_id', 'year', 'month'])['amount'].sum().groupby('prism_consumer_id').mean()
weekly_totals  = outflows.groupby(['prism_consumer_id', 'year', 'week'])['amount'].sum().groupby('prism_consumer_id').mean()
yearly_totals  = outflows.groupby(['prism_consumer_id', 'year', 'year'])['amount'].sum().groupby('prism_consumer_id').mean()

# display(outflows)

C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\822019574.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['year'] = outflows['posted_date'].dt.year
C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\822019574.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['month'] = outflows['posted_date'].dt.month
C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\822019574.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer

- Spending within 3 weeks, 6 months, etc.

In [11]:
outflows['posted_date'] = pd.to_datetime(outflows['posted_date'])
spending_over_time = outflows.sort_values(['prism_consumer_id', 'posted_date'])

C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\1537017176.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['posted_date'] = pd.to_datetime(outflows['posted_date'])


In [12]:
initial_dates = outflows.groupby('prism_consumer_id')['posted_date'].min()

In [13]:
spending_over_time = spending_over_time.merge(initial_dates, on='prism_consumer_id', how='left', suffixes=('', '_initial'))
spending_over_time = spending_over_time.rename(columns={'posted_date_initial': 'initial_date'})

In [14]:
spending_over_time['days_between'] = spending_over_time['posted_date'] - spending_over_time['initial_date']

spending_over_time['months_between'] = (
    (spending_over_time['posted_date'].dt.year - spending_over_time['initial_date'].dt.year) * 12 +
    (spending_over_time['posted_date'].dt.month - spending_over_time['initial_date'].dt.month)
).abs()

In [15]:
for weeks in range(7, 53, 7):    
    spending_over_time[f'first_{weeks}_weeks'] = spending_over_time['days_between'].astype('int64') <= weeks

for months in range(3, 13, 3):    
    spending_over_time[f'first_{months}_months'] = spending_over_time['months_between'] <= months

# spending_over_time

In [16]:
month_aggs = c_df[['prism_consumer_id']].drop_duplicates().reset_index(drop=True)

for months in range(3, 13, 3):  
    months_df = spending_over_time[spending_over_time[f'first_{months}_months']]
    
    agg_df = (
        months_df
        .groupby('prism_consumer_id')
        .agg(amount_sum=('amount', 'sum'), amount_std=('amount', 'std'), amount_mean=('amount', 'mean'))
        .reset_index()
    )

    month_aggs = month_aggs.merge(
        agg_df, on='prism_consumer_id', how='left', suffixes=('', f'_first_{months}_months')
    )

month_aggs

,prism_consumer_id,amount_sum,amount_std,amount_mean,amount_sum_first_6_months,amount_std_first_6_months,amount_mean_first_6_months,amount_sum_first_9_months,amount_std_first_9_months,amount_mean_first_9_months,amount_sum_first_12_months,amount_std_first_12_months,amount_mean_first_12_months
0,0,8797.15,73.835391,42.704612,14908.41,70.470938,40.293000,14908.41,70.470938,40.293000,14908.41,70.470938,40.293000
1,1,10605.49,137.990627,84.843920,23098.37,172.961059,95.055021,23098.37,172.961059,95.055021,23098.37,172.961059,95.055021
2,2,11040.12,110.238192,51.589346,22334.58,211.458378,60.857166,22334.58,211.458378,60.857166,22334.58,211.458378,60.857166
3,3,8461.56,105.087958,76.230270,19846.01,276.834340,90.209136,19846.01,276.834340,90.209136,19846.01,276.834340,90.209136
4,4,4888.81,92.358957,68.856479,6288.24,92.880270,62.882400,8849.66,81.981091,55.658239,17509.71,116.561814,65.825977
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,13995,702.00,95.217932,58.500000,740.00,86.516445,49.333333,850.00,79.300290,47.222222,850.00,79.300290,47.222222
11996,13996,19892.10,180.376868,95.177512,43741.75,332.235037,106.169296,53982.27,299.534140,102.627890,53982.27,299.534140,102.627890
11997,13997,5239.39,1438.633599,1047.878000,7425.31,1207.119639,825.034444,7425.31,1207.119639,825.034444,7425.31,1207.119639,825.034444
11998,13998,5327.77,191.110764,64.972805,31577.92,784.528992,196.136149,45669.84,835.539391,213.410467,45669.84,835.539391,213.410467


- Visualization for balance changes over time:

In [17]:
balance_changes = t[t['is_before_balance_date'] == False].sort_values(['prism_consumer_id', 'posted_date'])
# balance_changes

In [18]:
balance_changes['amount'] = pd.to_numeric(balance_changes['amount'], errors='coerce')

In [19]:
initial_balance = acct_balance.to_dict()

In [20]:
from collections import defaultdict

changes = defaultdict(list)

for id, amount in balance_changes[['prism_consumer_id', 'amount']].values:
    changes[id].append(amount)

In [21]:
balance_change = defaultdict(list)
for i, v in changes.items():  
    if i in acct_balance.keys():
        v.insert(0, acct_balance[i])  
    updated_balance = np.cumsum(v)
    balance_change[i] = updated_balance

print(len(balance_change))

11601


## Getting stats for balance changes overtime:

In [22]:
balance_change = pd.DataFrame(
    [(k, v) for k, lst in balance_change.items() for v in lst], 
    columns=['prism_consumer_id', 'balance_changes']
)
# balance_change

In [23]:
stats_changes = balance_change.groupby(['prism_consumer_id']).agg({'balance_changes': ['mean', 'std']})
# stats_changes

## Required features:

In [27]:
feats = t_df.groupby(['prism_consumer_id', 'category']).agg({'amount': ['count', 'sum', 'std', 'mean', 'median']})
feats = feats.unstack(level=1)
feats.columns = ['_'.join(col).strip() for col in feats.columns.values]
feats = feats.fillna(0)
# feats = feats.reset_index()
# feats.head()

- Putting everything together:

In [28]:
result = c_df[['prism_consumer_id', 'DQ_TARGET']].drop_duplicates().reset_index(drop=True)

result['current_balance'] = result['prism_consumer_id'].map(current_balance)
result['balance_mean'] = result['prism_consumer_id'].map(stats_changes[('balance_changes', 'mean')])
result['balance_std'] = result['prism_consumer_id'].map(stats_changes[('balance_changes', 'std')])
result['avg_spending'] = result['prism_consumer_id'].map(avg_spending).fillna(0)
result['avg_monthly_outflow'] = result['prism_consumer_id'].map(monthly_totals).fillna(0)
result['avg_weekly_outflow'] = result['prism_consumer_id'].map(weekly_totals).fillna(0)
result['avg_yearly_outflow'] = result['prism_consumer_id'].map(yearly_totals).fillna(0)

mapped_feats = pd.DataFrame({col: result['prism_consumer_id'].map(feats[col]) for col in feats.columns})
outflow_feats = pd.DataFrame({col: result['prism_consumer_id'].map(month_aggs[col]) for col in month_aggs.columns})
result = pd.concat([result, mapped_feats, outflow_feats], axis=1)
# result

In [29]:
# result.isna().sum()

In [30]:
result = result.loc[:,~result.columns.duplicated()].copy()
result

,prism_consumer_id,DQ_TARGET,current_balance,balance_mean,balance_std,avg_spending,avg_monthly_outflow,avg_weekly_outflow,avg_yearly_outflow,amount_count_ACCOUNT_FEES,...,amount_mean,amount_sum_first_6_months,amount_std_first_6_months,amount_mean_first_6_months,amount_sum_first_9_months,amount_std_first_9_months,amount_mean_first_9_months,amount_sum_first_12_months,amount_std_first_12_months,amount_mean_first_12_months
0,0,0.0,-201.22,334.377359,986.039607,40.293000,2129.772857,573.400385,14908.410,0.0,...,42.704612,14908.41,70.470938,40.293000,14908.41,70.470938,40.293000,14908.41,70.470938,40.293000
1,1,0.0,5107.85,5008.010698,1196.388690,95.055021,3299.767143,855.495185,23098.370,0.0,...,84.843920,23098.37,172.961059,95.055021,23098.37,172.961059,95.055021,23098.37,172.961059,95.055021
2,2,0.0,3235.49,5022.325479,2504.559496,60.857166,3190.654286,797.663571,11167.290,0.0,...,51.589346,22334.58,211.458378,60.857166,22334.58,211.458378,60.857166,22334.58,211.458378,60.857166
3,3,0.0,10462.25,7278.493493,1939.603669,90.209136,2835.144286,708.786071,9923.005,0.0,...,76.230270,19846.01,276.834340,90.209136,19846.01,276.834340,90.209136,19846.01,276.834340,90.209136
4,4,0.0,-2149.05,-674.058730,659.354952,65.825977,2501.387143,795.895909,8754.855,0.0,...,68.856479,6288.24,92.880270,62.882400,8849.66,81.981091,55.658239,17509.71,116.561814,65.825977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,13995,0.0,1873.31,1539.005873,293.333878,47.222222,106.250000,53.125000,425.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11996,13996,0.0,11173.97,12404.749627,2230.389122,102.627890,5998.030000,1458.980270,26991.135,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11997,13997,0.0,2444.61,3371.028372,1236.834973,825.034444,1856.327500,1485.062000,7425.310,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11998,13998,0.0,24322.07,20413.493344,4331.819603,213.410467,5074.426667,1304.852571,22834.920,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Predicting:

In [31]:
pd.Series(result.columns).value_counts()

prism_consumer_id                 1
DQ_TARGET                         1
amount_mean_EDUCATION             1
amount_mean_ENTERTAINMENT         1
amount_mean_ESSENTIAL_SERVICES    1
                                 ..
amount_sum_PAYCHECK               1
amount_sum_PENSION                1
amount_sum_PETS                   1
amount_sum_REFUND                 1
amount_mean_first_12_months       1
Name: count, Length: 256, dtype: int64

In [32]:
result = result.fillna(0)

In [33]:
cids = c_df.prism_consumer_id.unique()

X = cids
y = c_df["DQ_TARGET"]

X_2, test_cids, y_2, y_test = train_test_split(
    X, y, 
    test_size=0.2, random_state=16
)
train_cids, valid_cids, y_train, y_val = train_test_split(
    X_2, y_2, 
    test_size=0.25, random_state=16
)

# Features and labels for train, valid, and test sets:
X_train = result[result.prism_consumer_id.isin(train_cids)]
y_train = X_train.DQ_TARGET

X_valid = result[result.prism_consumer_id.isin(valid_cids)]
y_valid = X_valid.DQ_TARGET

X_test  = result[result.prism_consumer_id.isin(test_cids)]
y_test  = X_test.DQ_TARGET

X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
X_valid.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
X_test.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)

len(X_train), len(X_valid), len(X_test)

C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\1919337731.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\1919337731.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_valid.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
C:\Users\bdion\AppData\Local\Temp\ipykernel_29924\1919337731.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/in

(7200, 2400, 2400)

In [34]:
clf = LogisticRegression(random_state=420, penalty=None).fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.91      1.00      0.95      2194
         1.0       0.33      0.00      0.01       206

    accuracy                           0.91      2400
   macro avg       0.62      0.50      0.48      2400
weighted avg       0.86      0.91      0.87      2400



c:\Users\bdion\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [35]:
metrics.roc_auc_score(y_test, y_pred)

0.50197139595189

In [36]:
y_pred.sum()

3.0

In [37]:
y_test.sum()

206.0

In [38]:
y_pred = clf.predict(X_valid)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

         0.0       0.91      1.00      0.95      2184
         1.0       0.00      0.00      0.00       216

    accuracy                           0.91      2400
   macro avg       0.45      0.50      0.48      2400
weighted avg       0.83      0.91      0.87      2400



In [39]:
metrics.roc_auc_score(y_valid, y_pred)

0.49862637362637363

In [40]:
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': clf.coef_[0]})

coefficients.sort_values('Coefficient', ascending=False)[:50]

,Feature,Coefficient
112,amount_std_DEPOSIT,0.000303
135,amount_std_PAYCHECK,0.000122
143,amount_std_TAX,0.000122
96,amount_sum_TAX,0.000114
60,amount_sum_BNPL,0.000091
4,avg_monthly_outflow,0.000081
59,amount_sum_BILLS_UTILITIES,0.000074
87,amount_sum_OVERDRAFT,0.000071
61,amount_sum_CHILD_DEPENDENTS,0.000069
225,amount_median_MISCELLANEOUS,0.000065


- Positive coefficients → Increase probability of the positive class.
- Negative coefficients → Decrease probability.

In [41]:
clf = LogisticRegression(random_state=420, penalty=None, class_weight='balanced').fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.96      0.68      0.80      2194
         1.0       0.17      0.68      0.27       206

    accuracy                           0.68      2400
   macro avg       0.56      0.68      0.53      2400
weighted avg       0.89      0.68      0.75      2400



c:\Users\bdion\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [42]:
metrics.roc_auc_score(y_test, y_pred)

0.6807356338115425

In [43]:
y_pred.sum()

838.0

In [44]:
y_test.sum()

206.0

In [45]:
y_pred = clf.predict(X_valid)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.69      0.80      2184
         1.0       0.17      0.63      0.27       216

    accuracy                           0.69      2400
   macro avg       0.56      0.66      0.54      2400
weighted avg       0.88      0.69      0.75      2400



In [46]:
metrics.roc_auc_score(y_valid, y_pred)

0.663970288970289

In [47]:
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': clf.coef_[0]})

coefficients.sort_values('Coefficient', ascending=False)[:50]

,Feature,Coefficient
60,amount_sum_BNPL,0.000141
112,amount_std_DEPOSIT,0.000132
4,avg_monthly_outflow,0.000086
5,avg_weekly_outflow,0.000078
96,amount_sum_TAX,0.000070
178,amount_mean_MISCELLANEOUS,0.000065
68,amount_sum_ESSENTIAL_SERVICES,0.000062
87,amount_sum_OVERDRAFT,0.000060
135,amount_std_PAYCHECK,0.000060
225,amount_median_MISCELLANEOUS,0.000059
